# Optimización de Crop4Test

Se optimiza la separación y el refinamiento conjunto: 20 iteraciones de `Crop3Test` y después ajuste de confianza/continuidad con restricción de mezcla. Las pruebas usan las mismas mezclas de entrenamiento y semillas. El objetivo predeterminado es maximizar `ssim`; también se puede elegir `acc_both` o `recon_bpsnr`. Las fuentes originales solo se usan para evaluar el resultado, no para refinarlo. La validación utiliza el conjunto de test y no realimenta el estudio.


In [3]:
import importlib
import sys
from pathlib import Path

import numpy as np
import tensorflow as tf
import optuna
import matplotlib.pyplot as plt

project_root = next(
    root for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "project" / "CROP" / "models").is_dir()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from project.loader import Loader
import project.CROP.models.crop3_test as crop3_test_module
import project.CROP.models.crop4_test as crop4_test_module
importlib.reload(crop3_test_module)
importlib.reload(crop4_test_module)

SEED = 3333
DATASET = "fashion"
CVAE_NAME = "int_256_lat_64_beta_0_4_fashion.keras"
PREDICTOR_NAME = f"CCE_Conv2D_{DATASET}.keras"
N_IMAGES = 1000
N_TRIALS = 100
N_VALIDATION_IMAGES = 1000
N_VALIDATION_RUNS = 3
TARGET_METRIC = "ssim"  # Maximizar el SSIM medio de las estimaciones.
SEPARATION_ITERATIONS = 20
REFINEMENT_ITERATIONS = 80
assert TARGET_METRIC in {"acc_both", "ssim", "recon_bpsnr"}

FIXED_PARAMS = {
    "alpha_1": -2,
    "alpha_2": -22,
    "bias": 0.22,
    "slope": 22,
    "beta": 1.0,
    "alpha_mix": 0.5,
    "saturation_object_threshold": 0.05,
    "separation_iterations": SEPARATION_ITERATIONS,
    "refinement_iterations": REFINEMENT_ITERATIONS,
    "refinement_step": 0.05,
    "refinement_tolerance": 1e-5,
}

tf.keras.utils.set_random_seed(SEED)
data = Loader.data(dataset=DATASET)
cvae = Loader.cvae(model_name=CVAE_NAME)
predictor = Loader.predictor(model_name=PREDICTOR_NAME)

# Generador independiente de las semillas del CVAE y de Optuna.
rng = np.random.default_rng(SEED)
idx0 = rng.permutation(len(data["x_train"]))[:N_IMAGES]
idx1 = rng.permutation(len(data["x_train_1"]))[:N_IMAGES]
train_batch = (
    data["x_train"][idx0], data["x_train_1"][idx1],
    data["y_train"][idx0], data["y_train_1"][idx1],
)


def scalar_metric(metrics, key):
    value = metrics[key]
    # Las métricas de calidad devuelven (media, desvío).
    if isinstance(value, (tuple, list)):
        value = value[0]
    return float(value)


def objective(trial):
    params = {
        **FIXED_PARAMS,
        "gamma": trial.suggest_float("gamma", 0.0, 2.0),
        "delta": trial.suggest_float("delta", 0.0, 1.0),
        "refinement_correction_weight": trial.suggest_float(
            "refinement_correction_weight", 0.05, 5.0, log=True
        ),
        "refinement_continuity_weight": trial.suggest_float(
            "refinement_continuity_weight", 0.01, 1.0, log=True
        ),
        "refinement_signal_scale": trial.suggest_float(
            "refinement_signal_scale", 0.01, 0.2, log=True
        ),
        "refinement_edge_scale": trial.suggest_float(
            "refinement_edge_scale", 0.03, 0.3, log=True
        ),
    }
    tf.keras.utils.set_random_seed(SEED)
    model = crop4_test_module.Crop4Test(cvae=cvae, predictor=predictor)
    metrics = model.unmix(
        *train_batch,
        params=params,
        iterations=SEPARATION_ITERATIONS,
        labels=data["labels"],
        show_image=False,
        show_metrics=False,
    )
    score = scalar_metric(metrics, TARGET_METRIC)
    if not np.isfinite(score):
        raise ValueError(f"Métrica no finita: {TARGET_METRIC}={score}")
    trial.set_user_attr("effective_params", params)
    trial.set_user_attr("refinement_report", model.refinement_report)
    for key in ("acc_both", "ssim", "recon_bpsnr"):
        trial.set_user_attr(key, scalar_metric(metrics, key))
    return score


Usando fashion como dataset


## Estudio

Se optimizan gamma/delta y cuatro parámetros del refinamiento. Los pasos de separación y refinamiento quedan fijos para comparar el mismo presupuesto. No se guardan JSON de las imágenes por cada prueba.

In [4]:
study = optuna.create_study(
    direction="maximize",
    study_name=f"Crop4Test_{DATASET}_{TARGET_METRIC}",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study.optimize(objective, n_trials=N_TRIALS, n_jobs=1)
best_params = {**FIXED_PARAMS, **study.best_params}
print("Mejor valor:", study.best_value)
print("Parámetros completos:", best_params)
print("Refinamiento:", study.best_trial.user_attrs["refinement_report"])


[I 2026-11-11 21:21:39,237] A new study created in memory with name: Crop4Test_fashion_ssim
[I 2026-11-11 21:21:48,460] Trial 0 finished with value: 0.6999427676200867 and parameters: {'gamma': 1.5031512144089574, 'delta': 0.10925127538854273, 'refinement_correction_weight': 0.4690428957971171, 'refinement_continuity_weight': 0.09992228469385059, 'refinement_signal_scale': 0.07160112057499016, 'refinement_edge_scale': 0.05164234634004442}. Best is trial 0 with value: 0.6999427676200867.
[I 2026-11-11 21:21:57,728] Trial 1 finished with value: 0.7013279795646667 and parameters: {'gamma': 1.2255897813211882, 'delta': 0.11965240339743843, 'refinement_correction_weight': 1.3285398862559574, 'refinement_continuity_weight': 0.05560592685682976, 'refinement_signal_scale': 0.048165853869592004, 'refinement_edge_scale': 0.055005781343828004}. Best is trial 1 with value: 0.7013279795646667.
[I 2026-11-11 21:22:06,842] Trial 2 finished with value: 0.7116664052009583 and parameters: {'gamma': 1.24

Mejor valor: 0.71938157081604
Parámetros completos: {'alpha_1': -2, 'alpha_2': -22, 'bias': 0.22, 'slope': 22, 'beta': 1.0, 'alpha_mix': 0.5, 'saturation_object_threshold': 0.05, 'separation_iterations': 20, 'refinement_iterations': 80, 'refinement_step': 0.05, 'refinement_tolerance': 1e-05, 'gamma': 1.2264220308880407, 'delta': 0.48852505882424613, 'refinement_correction_weight': 0.050168233031677056, 'refinement_continuity_weight': 0.9579586881986802, 'refinement_signal_scale': 0.01180408928719608, 'refinement_edge_scale': 0.1751970975021949}
Refinamiento: {'iterations': 80, 'initial_objective': 2115.294921875, 'final_objective': 2018.83544921875, 'mixture_max_error_before': 0.18152529001235962, 'mixture_max_error_after': 2.9802322387695312e-08, 'step': 0.05}


## Visualización de la optimización

Gráficos con Matplotlib: no requieren Plotly ni instalaciones dentro del notebook.

In [ ]:
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
fig, ax = plt.subplots(figsize=(9, 4))
numbers = [t.number for t in completed]
values = np.array([t.value for t in completed])
ax.scatter(numbers, values, label="Prueba")
ax.plot(numbers, np.maximum.accumulate(values), label="Mejor hasta ahora", color="tab:orange")
ax.set_xlabel("Prueba")
ax.set_ylabel(TARGET_METRIC)
ax.grid(alpha=0.3)
ax.legend()
plt.show()
plt.close(fig)

names = list(study.best_params)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, name in zip(axes.flat, names):
    ax.scatter([t.params[name] for t in completed], values, alpha=0.7)
    if name.startswith("refinement_"):
        ax.set_xscale("log")
    ax.set_xlabel(name)
    ax.set_ylabel(TARGET_METRIC)
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()
plt.close(fig)


## Validación de Crop4Test

Ejecuta los mejores parámetros sobre mezclas de test con semillas distintas del entrenamiento. La primera corrida muestra imágenes, labels predichos, curvas y mapas de sospecha. Las restantes permiten medir variación entre mezclas.

In [ ]:
best_params = {**FIXED_PARAMS, **study.best_params}
validation_results = []
validation_rng = np.random.default_rng(SEED + 1000)
for run in range(N_VALIDATION_RUNS):
    idx0 = validation_rng.permutation(len(data["x_test"]))[:N_VALIDATION_IMAGES]
    idx1 = validation_rng.permutation(len(data["x_test_1"]))[:N_VALIDATION_IMAGES]
    tf.keras.utils.set_random_seed(SEED + 1000 + run)
    validation_model = crop4_test_module.Crop4Test(cvae=cvae, predictor=predictor)
    metrics = validation_model.unmix(
        data["x_test"][idx0], data["x_test_1"][idx1],
        data["y_test"][idx0], data["y_test_1"][idx1],
        params=best_params.copy(),
        iterations=SEPARATION_ITERATIONS,
        labels=data["labels"],
        # Mostrar una grilla de 1000 imagenes seria ilegible. Las imagenes
        # de la primera corrida se muestran abajo con un subconjunto.
        show_image=False,
        show_metrics=False,
        show_all_curves=(run == 0),
    )
    validation_results.append({
        "run": run + 1,
        **{key: scalar_metric(metrics, key) for key in ("acc_both", "ssim", "recon_bpsnr")},
        "refinement_report": validation_model.refinement_report.copy(),
    })
    print(validation_results[-1])
    if run == 0:
        validation_model.show_refinement_maps()
        first_validation_model = validation_model
        first_validation_indices = (idx0, idx1)

# Mostrar solo 10 mezclas de la primera corrida, con sus labels finales.
from project.helper.graphics.graphics import Graphics
m = first_validation_model
idx0, idx1 = first_validation_indices
limit = min(10, len(idx0), len(idx1))
Graphics.complete_plot(
    tf.convert_to_tensor(m.mixed_input[:limit]),
    tf.convert_to_tensor(data["x_test"][idx0[:limit]]),
    tf.convert_to_tensor(data["x_test_1"][idx1[:limit]]),
    data["y_test"][idx0[:limit]], data["y_test_1"][idx1[:limit]],
    tf.convert_to_tensor(m.source1_estimation[:limit]),
    tf.convert_to_tensor(m.source2_estimation[:limit]),
    tf.convert_to_tensor(m.mask1[:limit]), tf.convert_to_tensor(m.mask2[:limit]),
    m.predictions1[:limit], m.predictions2[:limit],
    tf.convert_to_tensor(m.source1_estimation[:limit]),
    model_params=best_params,
    class_labels=data["labels"],
)


## Resultados de validación

In [ ]:
validation_summary = {}
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, key in zip(axes, ("acc_both", "ssim", "recon_bpsnr")):
    values = np.array([row[key] for row in validation_results])
    validation_summary[key] = {"mean": float(values.mean()), "std": float(values.std())}
    ax.plot(np.arange(1, len(values) + 1), values, "o-")
    ax.axhline(values.mean(), color="tab:orange", linestyle="--")
    ax.set_title(f"{key}: {values.mean():.4f} ± {values.std():.4f}")
    ax.set_xlabel("Corrida de validación")
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()
plt.close(fig)
print(validation_summary)


## Guardar estudio y resultados

Guarda parámetros completos, todas las pruebas y validación en un archivo pickle; no guarda batches de imágenes.

In [ ]:
import pickle
from datetime import datetime

output_dir = project_root / "project" / "experiments" / "optim_params" / "results"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"crop4_test_{DATASET}_{TARGET_METRIC}_{datetime.now():%Y%m%d_%H%M%S}.pkl"
with output_path.open("wb") as stream:
    pickle.dump({
        "model": "Crop4Test",
        "dataset": DATASET,
        "cvae": CVAE_NAME,
        "predictor": PREDICTOR_NAME,
        "seed": SEED,
        "objective": TARGET_METRIC,
        "best_params": best_params,
        "study": study,
        "validation_results": validation_results,
        "validation_summary": validation_summary,
    }, stream)
print("Guardado:", output_path)
